# Modelo Vectorial y Evaluación (TF-IDF y Similitud del Coseno)
En este cuaderno implementaremos un sistema de Recuperación de la Información basado en el **modelo vectorial (TF-IDF)** utilizando `scikit-learn` y `NLTK`. 
Además, implementaremos la métrica de evaluación **MAP (Mean Average Precision)** utilizando las 20 necesidades de información de nuestro corpus.

*Nota: Se han adaptado las estructuras de datos (como `PlaintextCorpusReader`) y el paso del `tokenizer` para asimilar el estilo de código a las prácticas de la asignatura.*


In [22]:
import os
import pandas as pd
import numpy as np
import nltk
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Descargamos los recursos necesarios de NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

## 1. Carga del Corpus con NLTK
Utilizamos `PlaintextCorpusReader` tal y como se recomienda en las prácticas de NLTK para leer cómodamente todos los documentos `.txt` de nuestro directorio.


In [23]:
# Definimos el lector del corpus apuntando a nuestra carpeta
corpus_pokemon = PlaintextCorpusReader(
    root='../corpus',
    fileids=r'.*\.txt',
    encoding='utf-8'
)

# Extraemos los nombres de archivo y el contenido en bruto de cada uno
doc_names = [f.replace('.txt', '') for f in corpus_pokemon.fileids()]
documents = [corpus_pokemon.raw(f) for f in corpus_pokemon.fileids()]

print(f"Se han cargado {len(documents)} documentos del corpus usando PlaintextCorpusReader.")


Se han cargado 50 documentos del corpus usando PlaintextCorpusReader.


c:\Users\adrmu\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\corpus\reader\api.py:218: RuntimeWarning: Security Violation [CorpusReader]: Unauthorized path C:\Users\adrmu\Desktop\IA\Proyecto\IA-2526-Info-7\corpus\amoonguss.txt
  with self.open(f) as fp:
c:\Users\adrmu\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\data.py:388: RuntimeWarning: Security Violation [pathsec.open]: Unauthorized path C:\Users\adrmu\Desktop\IA\Proyecto\IA-2526-Info-7\corpus\amoonguss.txt
  stream = _secure_open(self._path, "rb")
c:\Users\adrmu\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\corpus\reader\api.py:218: RuntimeWarning: Security Violation [CorpusReader]: Unauthorized path C:\Users\adrmu\Desktop\IA\Proyecto\IA-2526-Info-7\corpus\azumarill.txt
  with self.open(f) as fp:
c:\Users\adrmu\AppData\Local\Programs\Python\Python312\Lib\site-packages\nltk\data.py:388: RuntimeWarning: Security Violation [pathsec.open]: Unauthorized path C:\Users\adrmu\Desktop\IA

## 2. Preprocesamiento y Construcción del Modelo TF-IDF
Definimos un `tokenizer` personalizado que integra la separación en tokens (`word_tokenize`), el paso a minúsculas, el filtrado de *stopwords* y el *stemming* (extracción de raíces léxicas). Luego, pasamos esta función directamente al `TfidfVectorizer` al estilo de la asignatura.


In [24]:
# Inicializamos el stemmer y las stopwords en inglés
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def tokenizador_pokemon(text):
    """Tokeniza, limpia stopwords/puntuación y extrae raíces léxicas."""
    tokens = word_tokenize(text)
    clean_tokens = [
        stemmer.stem(token.lower()) for token in tokens 
        if token.isalpha() and token.lower() not in stop_words
    ]
    return clean_tokens

# Creamos y ajustamos el modelo vectorial pasando nuestro tokenizador
vectorizador = TfidfVectorizer(
    token_pattern=None,        # Anula el patrón por defecto para forzar el uso de nuestro tokenizer
    tokenizer=tokenizador_pokemon
)

tfidf_matrix = vectorizador.fit_transform(documents)

print(f"Tamaño del vocabulario: {len(vectorizador.get_feature_names_out())} términos.")
print(f"Dimensiones de la matriz TF-IDF: {tfidf_matrix.shape}")


Tamaño del vocabulario: 3405 términos.
Dimensiones de la matriz TF-IDF: (50, 3405)


## 3. Recuperación mediante Similitud del Coseno
Para una consulta dada, calcularemos su representación en el mismo espacio TF-IDF y utilizaremos la similitud del coseno para encontrar los documentos más cercanos.


In [25]:
def buscar_tfidf(query, top_n=None):
    # Vectorizar la consulta (el vectorizador ya le aplicará el tokenizador_pokemon)
    query_vec = vectorizador.transform([query])
    
    # Calcular similitud del coseno entre la consulta y todos los documentos
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Ordenar los documentos de mayor a menor similitud
    related_docs_indices = cosine_sim.argsort()[::-1]
    
    resultados = []
    for i in related_docs_indices:
        similitud = cosine_sim[i]
        if similitud > 0: # Solo devolvemos documentos con alguna similitud
            resultados.append({
                'documento': doc_names[i],
                'similitud': similitud
            })
            
    if top_n is not None:
        resultados = resultados[:top_n]
        
    return resultados

# Prueba del buscador
consulta_prueba = "Swords Dance priority sweeper"
resultados = buscar_tfidf(consulta_prueba, top_n=5)

print(f"Resultados para la búsqueda: '{consulta_prueba}'")
for r in resultados:
    print(f"- {r['documento']}: {r['similitud']:.4f}")


Resultados para la búsqueda: 'Swords Dance priority sweeper'
- ogerpon-wellspring: 0.1033
- kingambit: 0.1014
- volcarona: 0.0891
- iron-boulder: 0.0854
- breloom: 0.0812


## 4. Interfaz Textual Interactiva
Bucle interactivo para buscar en el corpus introduciendo texto por consola (descomentar para usar).


In [28]:
def interfaz_busqueda():
     print("=== Buscador Pokémon TF-IDF ===")
     print("Escribe 'salir' para terminar.")
     while True:
         query = input("\nIntroduce tu búsqueda: ")
         if query.lower() == 'salir':
             break
         resultados = buscar_tfidf(query, top_n=5)
         print("\n--- Resultados Principales ---")
         if not resultados:
             print("No se encontraron documentos relevantes.")
         for i, r in enumerate(resultados, 1):
             print(f"{i}. {r['documento']} (Similitud: {r['similitud']:.4f})")
interfaz_busqueda()


=== Buscador Pokémon TF-IDF ===
Escribe 'salir' para terminar.

--- Resultados Principales ---
1. corviknight (Similitud: 0.1909)
2. garganacl (Similitud: 0.1778)
3. gliscor (Similitud: 0.1580)
4. ting-lu (Similitud: 0.1485)
5. rotom-wash (Similitud: 0.1403)


## 5. Evaluación del Sistema (MAP, Precisión y Sensibilidad)
En esta sección calcularemos las métricas de evaluación ejecutando las consultas de texto libre y comparándolas con los juicios de relevancia anotados en el CSV. Se calculará:
- **MAP** (Mean Average Precision)
- **Precisión Media**
- **Sensibilidad Media (Recall)**


In [27]:
def calcular_metricas_completas(retrieved_docs, relevant_docs):
    if not relevant_docs:
        return 0.0, 0.0, 0.0
    
    # 1. Average Precision (AP)
    relevant_count = 0
    precision_sum = 0.0
    for i, doc in enumerate(retrieved_docs, 1):
        if doc in relevant_docs:
            relevant_count += 1
            precision_at_i = relevant_count / i
            precision_sum += precision_at_i
    
    ap = precision_sum / len(relevant_docs) if relevant_count > 0 else 0.0
    
    # 2. Precisión y Sensibilidad (Recall) sobre todo el conjunto recuperado
    set_recuperados = set(retrieved_docs)
    set_relevantes = set(relevant_docs)
    aciertos = set_recuperados.intersection(set_relevantes)
    
    precision = len(aciertos) / len(set_recuperados) if set_recuperados else 0.0
    sensibilidad = len(aciertos) / len(set_relevantes) if set_relevantes else 0.0
    
    return ap, precision, sensibilidad

# Cargamos las necesidades de información
import pandas as pd
import numpy as np
df_necesidades = pd.read_csv('../necesidades_info/necesidades.csv')

lista_ap = []
lista_precision = []
lista_sensibilidad = []

print("\nEvaluando consultas vectoriales...")
for index, row in df_necesidades.iterrows():
    consulta_libre = row['consulta_texto_libre']
    relevantes = [doc.strip() for doc in row['relevantes'].split(',')]
    
    # Recuperamos documentos con similitud > 0
    resultados_busqueda = buscar_tfidf(consulta_libre)
    documentos_recuperados = [r['documento'] for r in resultados_busqueda]
    
    ap, precision, sensibilidad = calcular_metricas_completas(documentos_recuperados, relevantes)
    
    lista_ap.append(ap)
    lista_precision.append(precision)
    lista_sensibilidad.append(sensibilidad)
    
    print(f"Q{row['id']} - AP: {ap*100:6.2f}% | Prec: {precision*100:6.2f}% | Sens: {sensibilidad*100:6.2f}% | Consulta: {consulta_libre}")

# Calculamos los promedios globales
MAP = np.mean(lista_ap)
mean_precision = np.mean(lista_precision)
mean_sensibilidad = np.mean(lista_sensibilidad)

print("\n" + "="*50)
print(f"MEAN AVERAGE PRECISION (MAP): {MAP*100:6.2f}%")
print(f"PRECISIÓN GLOBAL PROMEDIO:    {mean_precision*100:6.2f}%")
print(f"SENSIBILIDAD GLOBAL PROMEDIO: {mean_sensibilidad*100:6.2f}%")
print("="*50)



Evaluando consultas vectoriales...
Q1 - AP: 0.4977 | Prec: 0.1400 | Sens: 1.0000 | Consulta: Swords Dance sweeper priority
Q2 - AP: 0.4583 | Prec: 0.0800 | Sens: 1.0000 | Consulta: defensive pivot Defog Rapid Spin hazard removal
Q3 - AP: 0.2017 | Prec: 0.0800 | Sens: 1.0000 | Consulta: fast special attacker Fairy resistance
Q4 - AP: 0.8819 | Prec: 0.1200 | Sens: 1.0000 | Consulta: Stealth Rock physical wall defensive setter
Q5 - AP: 0.2765 | Prec: 0.0800 | Sens: 1.0000 | Consulta: Knock Off special wall defensive utility
Q6 - AP: 0.5073 | Prec: 0.1200 | Sens: 1.0000 | Consulta: bulky Water type reliable recovery
Q7 - AP: 1.0000 | Prec: 0.0400 | Sens: 1.0000 | Consulta: Spore sleep status user
Q8 - AP: 0.1191 | Prec: 0.0600 | Sens: 1.0000 | Consulta: fast physical Dark Fighting attacker
Q9 - AP: 1.0000 | Prec: 0.0200 | Sens: 1.0000 | Consulta: Magic Bounce hazard control
Q10 - AP: 0.8167 | Prec: 0.0800 | Sens: 1.0000 | Consulta: Sun weather setter abuser Fire Protosynthesis
Q11 - AP: 0